# Comparison of ResNet8 networks trained on CIFAR10 with and without DP

## Imports

In [ ]:
import os

import torch
import torchvision
import torchvision.transforms as transform

from torchvision.datasets import CIFAR10
from torchvision import models
import torch.nn as nn
import torch.optim as optim


In [ ]:
# hyperparamètres
MAX_GRAD_NORM = 1.2
EPSILON = 50.0
DELTA = 1e-5
EPOCHS = 20

LR = 1e-3

BATCH_SIZE = 512
MAX_PHYSICAL_BATCH_SIZE = 128

## Dataset : CIFAR10

In [12]:
DATA_ROOT = './cifar10'

train_dataset = CIFAR10(
    root=DATA_ROOT, train=True, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
)

test_dataset = CIFAR10(
    root=DATA_ROOT, train=False, download=True, transform=transform)

test_loader = torch.utils.data.DataLoader(  
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

100.0%


## Models

In [ ]:
model = models.resnet18(num_classes=10)

# BatchNorm incompatible avec DP-SGD
from opacus.validators import ModuleValidator
errors = ModuleValidator.validate(model, strict=False)
errors[-5:]

In [ ]:
# donc on utilise fix pour remplacer les BatchNorm par des GroupNorm
model = ModuleValidator.fix(model)
ModuleValidator.validate(model, strict=False)

In [ ]:
# savoir si gpu disponible
print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nom du GPU :", torch.cuda.get_device_name(0))

# # utilise le GPU si disponible, sinon le CPU
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = model.to(device)

## Training